# 06 · Evaluation
**CRISP-DM Phase 5 — Evaluation**

`notebooks/05_modeling.ipynb` produced a model and a strategy backtest. This notebook does not train
anything new — it checks that result against the two success criteria fixed *before any data was seen*,
in `docs/01_business_understanding.md`:

| # | Criterion | Exact wording |
|---|---|---|
| 1 | Business success (Sec. 2) | The agent's strategy beats buy-and-hold BTC on **Sharpe ratio**, net of fees, over the **same** test window |
| 2 | Data-mining success (Sec. 4) | The classifier beats **both** the majority and persistence baselines on **both** macro-F1 **and** ROC-AUC, on the held-out test set |

That order matters: a result is only evaluated against the criteria the business committed to in advance,
not against whichever metric happens to look best in hindsight.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [2]:
import pandas as pd

from src import config
from src.evaluation.baselines import buy_and_hold, majority_baseline, persistence_baseline
from src.evaluation.criteria import (business_criterion_met, data_mining_criterion_met,
                                     hard_prediction_scores, macro_roc_auc)
from src.evaluation.strategy import backtest
from src.modeling.train import evaluate, make_models, split_X_y

CLASSES = [-1.0, 0.0, 1.0]

train = pd.read_parquet(config.PROCESSED_DIR / "btcusdt_1D_train.parquet")
test = pd.read_parquet(config.PROCESSED_DIR / "btcusdt_1D_test.parquet")
X_train, y_train = split_X_y(train)
X_test, y_test = split_X_y(test)

# Re-fit the model selected in 05_modeling.ipynb (random forest, chosen on val macro-F1).
model = make_models(random_state=0)["random_forest"]
model.fit(X_train, y_train)
pred_test = pd.Series(model.predict(X_test), index=X_test.index)
proba_test = model.predict_proba(X_test)

## 1 · Data-mining success criterion

Macro-F1 was already computed in `05_modeling.ipynb`. ROC-AUC was not — it is required by `docs/01` but
easy to skip, since accuracy/F1 feel like "the" metrics. Computing it here closes that gap. Baselines don't
output probabilities, so they're scored on their hard (one-hot) predictions — a fair, if blunt,
comparison: it tells us whether the model's probability *ranking* carries more information than a baseline's
single guess.

In [3]:
maj = majority_baseline(y_test)
per = persistence_baseline(y_test)
maj_pred = pd.Series(maj["class"], index=y_test.index)
per_pred = y_test.shift(1)
per_mask = y_test.notna() & per_pred.notna()

results = pd.DataFrame({
    "majority":    {"macro_f1": maj["macro_f1"],
                     "roc_auc": macro_roc_auc(y_test, hard_prediction_scores(maj_pred, CLASSES), CLASSES)},
    "persistence": {"macro_f1": per["macro_f1"],
                     "roc_auc": macro_roc_auc(y_test[per_mask],
                                               hard_prediction_scores(per_pred[per_mask], CLASSES), CLASSES)},
    "random_forest": {"macro_f1": evaluate(model, X_test, y_test)["macro_f1"],
                       "roc_auc": macro_roc_auc(y_test, proba_test, CLASSES)},
}).T
results.style.format("{:.3f}")

,macro_f1,roc_auc
majority,0.207,0.500
persistence,0.356,0.518
random_forest,0.302,0.579


In [4]:
dm_met = data_mining_criterion_met(
    model_f1=results.loc["random_forest", "macro_f1"], model_auc=results.loc["random_forest", "roc_auc"],
    majority_f1=results.loc["majority", "macro_f1"], majority_auc=results.loc["majority", "roc_auc"],
    persistence_f1=results.loc["persistence", "macro_f1"], persistence_auc=results.loc["persistence", "roc_auc"],
)
print(f"Data-mining success criterion met? {dm_met}")
print(f"  beats majority on F1: {results.loc['random_forest','macro_f1'] > results.loc['majority','macro_f1']}"
      f"   beats majority on AUC: {results.loc['random_forest','roc_auc'] > results.loc['majority','roc_auc']}")
print(f"  beats persistence on F1: {results.loc['random_forest','macro_f1'] > results.loc['persistence','macro_f1']}"
      f"   beats persistence on AUC: {results.loc['random_forest','roc_auc'] > results.loc['persistence','roc_auc']}")

Data-mining success criterion met? False
  beats majority on F1: True   beats majority on AUC: True
  beats persistence on F1: False   beats persistence on AUC: True


**Reading this honestly:** the model's ROC-AUC (0.58) does beat both baselines — its predicted
probabilities rank the true class better than chance, which is a genuine (if modest) signal. But the
criterion requires winning on **both** metrics against **both** baselines, and macro-F1 fails against
persistence. A model can have better-than-chance probability ranking and still make worse hard decisions
than a trivial rule, if its decision threshold is miscalibrated — which is consistent with it defaulting
to "flat" far too often (see `docs/05_modeling.md`). **Criterion not met**, but not met for a specific,
diagnosable reason rather than "nothing works at all".

## 2 · Business success criterion

Reusing the strategy backtest from `05_modeling.ipynb`: predicted labels -> position -> realized return,
net of the 0.1%-per-side fee from `docs/01`, compared to buy-and-hold over the identical test window.

In [5]:
raw_1d = pd.read_parquet(config.INTERIM_DIR / f"{config.SYMBOL.lower()}_1D_clean.parquet")
test_close = raw_1d.loc[test.index.min():test.index.max(), "close"]

strategy_result = backtest(pred_test, test_close)
bh_result = buy_and_hold(test_close, "1D")
biz_met = business_criterion_met(strategy_result["sharpe"], bh_result["sharpe"])

compare = pd.DataFrame({"strategy": strategy_result, "buy-and-hold": bh_result}).T
print(f"Business success criterion met? {biz_met}")
compare.style.format({"total_return": "{:+.1%}", "sharpe": "{:.2f}", "max_drawdown": "{:.1%}"})

Business success criterion met? True


,total_return,sharpe,max_drawdown
strategy,+9.5%,0.44,-17.1%
buy-and-hold,-24.9%,-0.33,-53.0%


**Reading this honestly, per the risk `docs/01` Sec. 5 flagged in advance** ("regime change... the test
period must be reported with its regime"): this test window is a bear market (buy-and-hold Sharpe −0.33).
The strategy's win comes from predicting "flat" on 89% of days — it is in the market only ~11% of the time,
which mechanically avoids most of the drawdown. **Criterion technically met**, but the result would not
replicate confidence that the strategy has genuine skill; it has not been shown to win in a bull or
range-bound regime, only to avoid losing in this specific bear one.

## 3 · Overall verdict

| Criterion | Status | Caveat |
|---|---|---|
| Data-mining success (docs/01 Sec. 4) | **Not met** | ROC-AUC beats both baselines; macro-F1 does not beat persistence |
| Business success (docs/01 Sec. 2) | **Technically met** | Driven by low market exposure in a bear-market test window, not demonstrated directional skill |

**Recommendation: do not proceed to Deployment (Phase 6) on this model as-is.** The business criterion
passing on a single bear-market window is exactly the false positive `docs/01`'s own risk log warned about.
Before trusting this result:

1. **Re-test across regimes** — rerun the same chronological pipeline with a bull-market and a
   range-bound test window (the yearly Sharpe table in `03_eda.ipynb` item 12 shows which years qualify) to
   see if the business criterion still holds, or only holds in a downturn.
2. **Address the data-mining gap** — the model's AUC edge without a matching F1 edge points to a
   calibration/threshold problem, not a total absence of signal. Worth trying class-balanced decision
   thresholds or a cost-sensitive objective before concluding the feature set is exhausted.
3. Only after (1) and (2) should this feed into the full agent (news sentiment + market indicators) with
   any confidence that the ML leg contributes real signal rather than a favorable test window.